In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
working_directory = "/Users/kemalinecik/git_nosync/sctram"

In [3]:
import sys
sys.path.append(working_directory)

import logging
import os
import numpy as np
import pandas as pd
import networkx as nx
import scanpy as sc
import anndata as ad
from sctram.generate.real import sc_norman_sciplex_cpa

sc.settings.verbosity = 3

In [4]:
from sctram.api._lower import TrajectoryEvaluationAPI
from sctram.input import read_dict

2025-02-06 16:00:29.253 | INFO     | sctram.api._defaults_read:load_default_metrics:21 - Loaded default metrics from /Users/kemalinecik/git_nosync/sctram/sctram/api/_defaults.yaml
2025-02-06 16:00:29.254 | INFO     | sctram.api._defaults_read:load_default_metrics:74 - Default metrics YAML structure validated successfully.


In [5]:
dataset_dir = "/Users/kemalinecik/git_nosync/sctram/__temp__/data"
adata_norman = sc_norman_sciplex_cpa(dataset_dir=dataset_dir)
adata_bms = adata_norman[["bms" in i.lower() or "vehicle" in i.lower() for i in adata_norman.obs["drug"]]]
adata = ad.AnnData(X=adata_bms.obsm["tardis"].copy(), obs=adata_bms.obs.copy())

2025-02-06 16:00:29.286 | WARNING  | sctram.generate.real._download:download_dataset:105 - File PosixPath('/Users/kemalinecik/git_nosync/sctram/__temp__/data/norman_sciplex_cpa.h5ad') already exists. Skipping download.


In [8]:
ground_truth_trajectories = {
    "trajectory_1": [
        ('Vehicle_1.0', 'BMS_0.001'),
        ('BMS_0.001', 'BMS_0.005'),
        ('BMS_0.005', 'BMS_0.01'),
        ('BMS_0.01', 'BMS_0.05'),
        ('BMS_0.05', 'BMS_0.1'),
        ('BMS_0.1', 'BMS_0.5'),
        ('BMS_0.5', 'BMS_1.0'),
    ],
}
input_trajectories_all = read_dict(ground_truth_trajectories)
input_trajectories = input_trajectories_all.get_trajectory("trajectory_1", include_additional_nodes=False)

In [10]:
api = TrajectoryEvaluationAPI(
    adata=adata,
    input_trajectories=input_trajectories,
    labels_obs="drug_dose_name",
    root_label="Vehicle_1.0",
    logger_level="DEBUG"
)

In [11]:
api.evaluate_pseudotime()

2025-02-06 16:03:07.825 | INFO     | sctram.api._lower:evaluate_pseudotime:71 - Starting pseudotime evaluation.
2025-02-06 16:03:07.825 | INFO     | sctram.api._lower:evaluate_pseudotime:76 - Running pseudotime inference with method 'DPTInference'
2025-02-06 16:03:07.826 | INFO     | sctram.api._lower:evaluate_pseudotime:81 - Running pseudotime evaluation with method 'PseudotimeValuesEvaluation'
2025-02-06 16:03:07.827 | DEBUG    | sctram.infer._base:_initialize_from_adata_without_neighbors:160 - Initializing from AnnData without precomputed neighbors.
2025-02-06 16:03:07.829 | DEBUG    | sctram.infer._base:_add_labels_to_adata:219 - Adding provided labels to AnnData object.
2025-02-06 16:03:07.830 | INFO     | sctram.infer._base:_initialize_from_adata_without_neighbors:167 - No precomputed neighbors found in AnnData.
2025-02-06 16:03:07.830 | DEBUG    | sctram.infer._base:_initialize_from_adata_without_neighbors:171 - AnnData initialized successfully from AnnData without neighbors.
20

computing neighbors
    using data matrix X directly
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:03)


2025-02-06 16:03:10.911 | INFO     | sctram.infer._base:_needs_diffmap:520 - Computing Diffusion Map.


computing Diffusion Maps using n_comps=15(=n_dcs)
computing transitions
    finished (0:00:00)
    eigenvalues of transition matrix
    [1.         0.9957283  0.98897034 0.9752918  0.96000016 0.93977296
     0.9325454  0.9238255  0.90231395 0.8939387  0.89036936 0.88236076
     0.86557806 0.8639325  0.862859  ]
    finished: added
    'X_diffmap', diffmap coordinates (adata.obsm)
    'diffmap_evals', eigenvalues of transition matrix (adata.uns) (0:00:00)


2025-02-06 16:03:11.058 | DEBUG    | sctram.infer._base:_needs_diffmap:522 - Diffusion Map computed successfully.
2025-02-06 16:03:11.058 | DEBUG    | sctram.infer._dpt:_calculate:103 - Setting root using provided label-based specification.
2025-02-06 16:03:11.060 | INFO     | sctram.infer._dpt:_set_root_from_label_dict:301 - Outlier removal applied. 2155 out of 2155 cells remain after filtering based on z-score <= 3.
2025-02-06 16:03:11.060 | INFO     | sctram.infer._dpt:set_root_from_label_dict:231 - Root set to cell index 2575 based on minimum in DiffMap component 0 within label 'Vehicle_1.0'.
2025-02-06 16:03:11.060 | INFO     | sctram.infer._dpt:_calculate:119 - Performing DPT trajectory calculation.


computing Diffusion Pseudotime using n_dcs=10
    finished: added
    'dpt_pseudotime', the pseudotime (adata.obs) (0:00:00)


2025-02-06 16:03:11.065 | DEBUG    | sctram.infer._dpt:_calculate:121 - DPT calculation completed successfully.
2025-02-06 16:03:11.066 | INFO     | sctram.infer._base:calculate:502 - Trajectory inference completed successfully.
2025-02-06 16:03:11.066 | DEBUG    | sctram.evaluate._pseudotime:__init__:41 - Initialized PseudotimeEvaluation with metrics: ['pearson', 'spearman', 'kendall', 'mse', 'mae', 'r2', 'r2_with_spline', 'concordance_index', 'dynamic_time_warping', 'wasserstein_distance']
2025-02-06 16:03:11.067 | INFO     | sctram.evaluate._base:evaluate:159 - Starting evaluation with PseudotimeValuesEvaluation.
2025-02-06 16:03:11.067 | DEBUG    | sctram.evaluate._base:evaluate:162 - Verifying given trajectory.
2025-02-06 16:03:11.067 | DEBUG    | sctram.evaluate._base:evaluate:165 - Verifying inferred trajectory.
2025-02-06 16:03:11.068 | DEBUG    | sctram.evaluate._base:evaluate:169 - Verifying labels.
2025-02-06 16:03:11.068 | DEBUG    | sctram.evaluate._pseudotime:_verify_labe

Diffusion pseudotime converged in 21 steps.


2025-02-06 16:03:11.241 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate_concordance_index:282 - Concordant pairs: 5417089, Discordant pairs: 190320, Usable pairs: 5607409
2025-02-06 16:03:11.242 | INFO     | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate_concordance_index:294 - Calculated Concordance Index: 0.9660591906172709
2025-02-06 16:03:11.243 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate:44 - Calculating metric: 'dynamic_time_warping'
2025-02-06 16:03:11.244 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate_dynamic_time_warping:330 - 'fastdtw' library is not found. Using fallback DTW implementation
2025-02-06 16:03:22.137 | DEBUG    | sctram.evaluate._metricsmixin._pseudotimevaluesmetricsmixin:_calculate_dynamic_time_warping:372 - Fallback DTW distance: 0.11527846870098829
2025-02-06 16:03:22.139 | DEBUG    | sctram.evaluate._metricsmixin._pseudotim

In [12]:
api.results

{'pseudotime': {'pearson': 0.9022949861402323,
  'spearman': 0.8681034142097269,
  'kendall': 0.7625571244858326,
  'mse': 0.03735907642195845,
  'mae': 0.13085564787533857,
  'r2': 0.7830202444641525,
  'r2_with_spline': 0.8443289958775226,
  'concordance_index': 0.9660591906172709,
  'dynamic_time_warping': 0.11527846870098829,
  'wasserstein_distance': 0.1129031260820259}}